In [1]:
import torch
from torch import nn
from datasets import load_dataset
from transformers import GPT2Tokenizer
import math
from torch.utils.data import DataLoader, dataset
from tqdm import tqdm
from tokenizers import ByteLevelBPETokenizer

from coreTransformer import *
from tokenizer import *
from trainingSetup import *
from dataloader import *
from inferenceTest import *

In [2]:
def initialize_weights(model):
    """More conservative weight initialization"""
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight, gain=0.01)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.xavier_uniform_(module.weight, gain=0.01)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.ones_(module.weight)
            torch.nn.init.zeros_(module.bias)
    model.apply(_init_weights)
    return model

In [3]:
def initialize_model(tokenizer, device='cuda'):
    model = SmallTransformer(
        vocab_size=tokenizer.get_vocab_size(),
        d_model=256,
        num_heads=8,
        num_layers=6,
        d_ff=1024,
        max_seq_len=512,
        dropout=0.1
    )
    
    # Initialize weights properly
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            module.weight.data.normal_(mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)
    
    model.apply(_init_weights)
    return model.to(device)

In [ ]:
# Initialize model and training components
num_epochs = 50
train_loader, val_loader, tokenizer = get_dataloaders(batch_size=8, tokenizer_type='byte-level-bpe')

model = SmallTransformer(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=512,  # Increased model capacity
    num_heads=8,
    num_layers=8,  # More layers
    d_ff=2048,    # Larger feed-forward network
    max_seq_len=512,
    dropout=0.1
)

# Train the model
train_model(model, train_loader, val_loader, num_epochs=num_epochs, patience=3)



Starting training on cuda
Number of batches: 2971
Accumulation steps: 16
Effective batch size: 128


Validating: 100%|██████████| 308/308 [00:09<00:00, 34.09it/s]



Epoch 1:
Average training loss: 7.7276
Average validation loss: 4.4210
Learning rate: 1.00e-05
Saving best model...


Validating: 100%|██████████| 308/308 [00:09<00:00, 33.17it/s]



Epoch 2:
Average training loss: 2.9731
Average validation loss: 2.6147
Learning rate: 2.00e-05
Saving best model...


...


Epoch 28:
Average training loss: 1.5267
Average validation loss: 1.5726
Learning rate: 5.78e-05
Saving best model...


Validating: 100%|██████████| 308/308 [00:09<00:00, 33.92it/s]



Epoch 29:
Average training loss: 1.5065
Average validation loss: 1.5475
Learning rate: 5.39e-05
Saving best model...


Validating: 100%|██████████| 308/308 [00:09<00:00, 33.14it/s]



Epoch 30:
Average training loss: 1.4906
Average validation loss: 1.5298
Learning rate: 5.00e-05
Saving best model...


In [5]:

# _, _, tokenizer = get_dataloaders(batch_size=8, tokenizer_type='byte-level-bpe')

# model = SmallTransformer(
#     vocab_size=tokenizer.get_vocab_size(),
#     d_model=512,  # Increased model capacity
#     num_heads=8,
#     num_layers=8,  # More layers
#     d_ff=2048,    # Larger feed-forward network
#     max_seq_len=512,
#     dropout=0.1
# )
# print("\nTesting inference:")
# test_prompts = [
#     "In the distant future,",
#     "The scientist discovered",
#     "Once upon a time in",
# ]

# for prompt in test_prompts:
#     print(f"\nPrompt: {prompt}")
#     try:
#         generated = test_inference(
#             model=model,
#             tokenizer=tokenizer,
#             prompt=prompt,
#             max_new_tokens=50,
#             temperature=0.3,  # Higher temperature for more randomness
#             top_k=50
#         )
#         print(f"Generated: {prompt}{generated}")
#     except Exception as e:
#         print(f"Error generating text for prompt '{prompt}': {e}")

In [6]:
# Test with different parameters
if __name__ == "__main__":
    _, _, tokenizer = get_dataloaders(batch_size=8, tokenizer_type='byte-level-bpe')

    model = SmallTransformer(
        vocab_size=tokenizer.get_vocab_size(),
        d_model=512,  # Increased model capacity
        num_heads=8,
        num_layers=8,  # More layers
        d_ff=2048,    # Larger feed-forward network
        max_seq_len=512,
        dropout=0.1
    )
    test_prompts = [
        "In the distant future,",
        "The scientist discovered",
        "Once upon a time in",
    ]
    
    # Different parameter sets to try
    sampling_params = [
        {
            "temperature": 0.7,
            "top_k": 50,
            "top_p": 0.9,
            "repetition_penalty": 1.2,
            "max_new_tokens": 50
        },
        {
            "temperature": 0.9,
            "top_k": 40,
            "top_p": 0.85,
            "repetition_penalty": 1.3,
            "max_new_tokens": 50
        },
        {
            "temperature": 0.8,
            "top_k": 30,
            "top_p": 0.8,
            "repetition_penalty": 1.4,
            "max_new_tokens": 50
        }
    ]
    
    print("\nTesting inference with different parameters:")
    for prompt in test_prompts:
        print(f"\nPrompt: {prompt}")
        for params in sampling_params:
            try:
                print(f"\nParameters: {params}")
                generated = test_inference(model, tokenizer, prompt, **params)
                print(f"Generated: {prompt}{generated}")
            except Exception as e:
                print(f"Error generating text with parameters {params}: {e}")


Testing inference with different parameters:

Prompt: In the distant future,

Parameters: {'temperature': 0.7, 'top_k': 50, 'top_p': 0.9, 'repetition_penalty': 1.2, 'max_new_tokens': 50}
Model loaded successfully!
Generated: In the distant future, mph the time the time the time time the time time time time

Parameters: {'temperature': 0.9, 'top_k': 40, 'top_p': 0.85, 'repetition_penalty': 1.3, 'max_new_tokens': 50}
Model loaded successfully!
Generated: In the distant future, 000 $ 360 @,@ 000 ) and most successful , most expensive and the most popular and most important national scale . Most British and most known as South of major numbers and many among number and most commonly known in European records and most important and most important in

Parameters: {'temperature': 0.8, 'top_k': 30, 'top_p': 0.8, 'repetition_penalty': 1.4, 'max_new_tokens': 50}
Model loaded successfully!
Generated: In the distant future, Torres just one no one only one most more one more one very much two but 